In [0]:
from pyspark.sql.functions import (
    col, regexp_replace, trim, initcap, to_date, when, lit
)

bronze_df = spark.read.format("delta").table("final_project.Bronze_table")
silver_df = (
    bronze_df
    .withColumn("Price", regexp_replace(col("Price"), "[^0-9]", "").cast("bigint"))
    .withColumn("Mileage", regexp_replace(col("Mileage"), "[^0-9]", "").cast("double"))
    .withColumn("Year", col("Year").cast("int"))
    .withColumn("Engine_CC", regexp_replace(col("Engine_CC"), "[^0-9]", "").cast("int"))
    .withColumn("Brand", initcap(trim(col("Brand"))))
    .withColumn("Model", trim(col("Model")))
    .withColumn("Transmission", initcap(trim(col("Transmission"))))
    .withColumn("Fuel_type", initcap(trim(col("Fuel_type"))))
    .withColumn("Location", trim(col("Location")))
    .withColumn("Date_Posted", to_date(col("Date_Posted"), "yyyy-MM-dd"))
    .replace("Null", None)
)


In [0]:
silver_df = silver_df.dropDuplicates()
silver_df = silver_df.dropna(subset=["Brand", "Model", "Price"])
silver_df.display()


In [0]:
silver_df.tail(5)

In [0]:
from pyspark.sql.functions import col, first, coalesce, count, row_number
from pyspark.sql.window import Window


model_lookup = (
    bronze_df
    .filter(col("Engine_CC").isNotNull())
    .groupBy("Model")
    .agg(first("Engine_CC").alias("model_engine_cc"))
)

silver_df = (
    bronze_df.alias("b")
    .join(model_lookup.alias("m"), on="Model", how="left")
    .withColumn("Engine_CC", coalesce(col("b.Engine_CC"), col("m.model_engine_cc")))
    .drop("model_engine_cc")
)


brand_counts = (
    silver_df
    .filter(col("Engine_CC").isNotNull())
    .groupBy("Brand", "Engine_CC")
    .count()
)

window = Window.partitionBy("Brand").orderBy(col("count").desc())

brand_lookup = (
    brand_counts
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .select("Brand", col("Engine_CC").alias("brand_engine_cc"))
)


silver_df = (
    silver_df.alias("s")
    .join(brand_lookup.alias("b"), on="Brand", how="left")
    .withColumn("Engine_CC", coalesce(col("s.Engine_CC"), col("b.brand_engine_cc")))
    .drop("brand_engine_cc")
)



silver_df.select(count(col("Engine_CC")).alias("Non-null Engine_CC")).show()
silver_df.selectExpr("count(*) - count(Engine_CC) as Remaining_Nulls").show()

display(silver_df)

In [0]:
from pyspark.sql.functions import col, count, when

null_counts = silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
])

display(null_counts)

In [0]:
from pyspark.sql.functions import (
    col,
    first,
    coalesce,
    row_number,
    count,
    when
)
from pyspark.sql.window import Window


model_lookup = (
    silver_df
    .filter(col("Fuel_type").isNotNull())
    .groupBy("Model")
    .agg(first("Fuel_type").alias("model_fuel"))
)

silver_df = (
    silver_df.alias("s")
    .join(model_lookup.alias("m"), on="Model", how="left")
    .withColumn("Fuel_type", coalesce(col("s.Fuel_type"), col("m.model_fuel")))
    .drop("model_fuel")
)


brand_counts = (
    silver_df
    .filter(col("Fuel_type").isNotNull())
    .groupBy("Brand", "Fuel_type")
    .count()
)

window = Window.partitionBy("Brand").orderBy(col("count").desc())

brand_lookup = (
    brand_counts
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .select("Brand", col("Fuel_type").alias("Brand_fuel"))
)


silver_df = (
    silver_df.alias("s")
    .join(brand_lookup.alias("b"), on="Brand", how="left")
    .withColumn("Fuel_type", coalesce(col("s.Fuel_type"), col("b.Brand_fuel")))
    .drop("Brand_fuel")
)


silver_df.select(
    count(when(col("Fuel_type").isNull(), 1)).alias("Remaining Fuel Type Nulls")
).show()

In [0]:
from pyspark.sql.functions import col, count, when
silver_df = silver_df.fillna({
    "Link": "https://www.contactcars.com/en"
})

silver_df.select(
    count(when(col("Link").isNull(), 1)).alias("Remaining_Null_Links")
).show()

display(silver_df)

In [0]:

silver_df = silver_df.fillna({
    "Location": "Unknown"
})

silver_df.select(
    count(when(col("Location").isNull(), 1)).alias("Remaining_Null_Location")
).show()


In [0]:
from pyspark.sql.functions import col, count, when

null_counts = silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
])

display(null_counts)

In [0]:
silver_df = silver_df.na.drop(
    subset=["Fuel_type", "Engine_CC"]
)

In [0]:
silver_df.count()

In [0]:
display(silver_df)

In [0]:
from pyspark.sql.functions import countDistinct

for column in silver_df.columns:
    distinct_count = silver_df.select(countDistinct(column)).collect()[0][0]
    
    print(f"\n{'='*60}")
    print(f"Column: {column}")
    print(f"Distinct Values: {distinct_count}")
    print(f"{'='*60}")
    
    display(
        silver_df.select(column).distinct().orderBy(column)
    )

In [0]:
from pyspark.sql.functions import col, regexp_replace, lower, initcap
silver_df = (
    silver_df
    .withColumn(
        "Brand",
        initcap(
            lower(
                regexp_replace(col("Brand"), "\\s+", "")
            )
        )
    )
    .withColumn(
        "Model",
        initcap(
            lower(
                regexp_replace(col("Model"), "\\s+", "")
            )
        )
    )
)

In [0]:
from pyspark.sql.functions import col, regexp_replace

silver_df = (
    silver_df
    .withColumn(
        "Price",
        regexp_replace(col("Price"), "[^0-9]", "").cast("bigint")
    )
    .withColumn(
        "Mileage",
        regexp_replace(col("Mileage"), "[^0-9]", "").cast("int")
    )
)
display(silver_df)


In [0]:
from pyspark.sql.functions import col, when

silver_df = silver_df.withColumn(
    "State",
    when(col("Mileage") == 0, "New")
    .otherwise("Used")
)

display(silver_df)

In [0]:
silver_df = silver_df.filter((col("Price") >= 30000) & (col("Price") < 15000000))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS final_project")
silver_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("final_project.Silver_table")